# K-Fold Cross-Validation from Scratch

Every other notebook in this repo evaluates models on a single `train_test_split`. That
estimate is noisy: which 20% ends up in the test set changes the reported accuracy by a
few points, especially on small datasets. K-fold cross-validation fixes this by rotating
the held-out fold `k` times and averaging, giving a more stable estimate and letting us
pick hyperparameters without ever touching the final test set.

This notebook implements `k_fold_split` and a generic `cross_val_score`, then uses them to:
1. Show that k-fold estimates are less noisy than single-split estimates.
2. Select a regularisation strength for logistic regression by CV instead of by eyeballing the test set.
3. Sanity-check the result against sklearn's `cross_val_score`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score as sklearn_cv_score
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## `k_fold_split`

Shuffles the sample indices, then cuts them into `k` contiguous folds of near-equal size
(the first `n % k` folds absorb one extra sample so sizes never differ by more than one).
Each fold takes a turn as the validation set while the rest form the training set.

In [ ]:
def k_fold_split(n_samples, k=5, shuffle=True, seed=None):
    idx = np.arange(n_samples)
    if shuffle:
        rng = np.random.RandomState(seed)
        rng.shuffle(idx)

    fold_sizes = np.full(k, n_samples // k, dtype=int)
    fold_sizes[: n_samples % k] += 1

    folds = []
    start = 0
    for size in fold_sizes:
        val_idx = idx[start:start + size]
        train_idx = np.concatenate([idx[:start], idx[start + size:]])
        folds.append((train_idx, val_idx))
        start += size
    return folds


# Sanity check: every sample appears in exactly one validation fold, and
# train/val never overlap within a fold.
folds = k_fold_split(100, k=5, seed=0)
all_val = np.concatenate([v for _, v in folds])
assert sorted(all_val) == list(range(100))
assert all(len(set(tr) & set(val)) == 0 for tr, val in folds)
print('k_fold_split defined and verified.')

## Generic `cross_val_score`

Takes any `fit_predict_fn(X_train, y_train, X_val) -> y_pred` so it works with the
from-scratch logistic regression below, or with a Random Forest, KNN, etc. from other
notebooks in this repo — nothing here is specific to one model.

In [ ]:
def cross_val_score(X, y, k, fit_predict_fn,
                     metric_fn=lambda y_true, y_pred: np.mean(y_true == y_pred),
                     shuffle=True, seed=0):
    folds = k_fold_split(len(X), k=k, shuffle=shuffle, seed=seed)
    scores = []
    for train_idx, val_idx in folds:
        y_pred = fit_predict_fn(X[train_idx], y[train_idx], X[val_idx])
        scores.append(metric_fn(y[val_idx], y_pred))
    return np.array(scores)


print('cross_val_score defined.')

## A model to evaluate: from-scratch logistic regression

Standardised gradient descent with optional L2 regularisation, same formulation as
`Logistic Regression.ipynb`, wrapped as a single `fit_predict` call.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def train_logistic(X_train, y_train, lr=0.1, epochs=500, l2=0.0):
    n, d = X_train.shape
    mu, sigma = X_train.mean(axis=0), X_train.std(axis=0) + 1e-8
    Xs = (X_train - mu) / sigma
    W, b = np.zeros(d), 0.0
    for _ in range(epochs):
        yhat = sigmoid(Xs @ W + b)
        grad_w = (Xs.T @ (yhat - y_train)) / n + l2 * W
        grad_b = np.mean(yhat - y_train)
        W -= lr * grad_w
        b -= lr * grad_b
    return W, b, mu, sigma


def logistic_fit_predict(X_tr, y_tr, X_val, lr=0.1, epochs=500, l2=0.0):
    W, b, mu, sigma = train_logistic(X_tr, y_tr, lr=lr, epochs=epochs, l2=l2)
    Xs_val = (X_val - mu) / sigma
    return (sigmoid(Xs_val @ W + b) >= 0.5).astype(int)


print('Logistic regression helpers defined.')

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target.astype(float)

scores = cross_val_score(X, y, k=5, fit_predict_fn=logistic_fit_predict)
print('Fold accuracies:', np.round(scores, 4))
print(f'Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')

## Single split vs. k-fold: which estimate is more stable?

Run 20 different random 80/20 splits and 20 different 5-fold CV runs (each with a
different shuffle seed), and compare how much the reported accuracy jumps around.

In [ ]:
single_split_accs = []
for seed in range(20):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)
    pred = logistic_fit_predict(X_tr, y_tr, X_val)
    single_split_accs.append(np.mean(pred == y_val))

kfold_means = []
for seed in range(20):
    s = cross_val_score(X, y, k=5, fit_predict_fn=logistic_fit_predict, seed=seed)
    kfold_means.append(s.mean())

print(f'Single 80/20 split (20 seeds) -> mean={np.mean(single_split_accs):.4f}  std={np.std(single_split_accs):.4f}')
print(f'5-fold CV mean     (20 seeds) -> mean={np.mean(kfold_means):.4f}  std={np.std(kfold_means):.4f}')

plt.figure(figsize=(6, 4))
plt.boxplot([single_split_accs, kfold_means], tick_labels=['Single 80/20 split', '5-fold CV mean'])
plt.ylabel('Accuracy')
plt.title('Estimate Stability: Single Split vs. K-Fold CV')
plt.tight_layout(); plt.show()

## Using CV to pick a hyperparameter

Sweep the L2 strength on a development set using 5-fold CV, pick the value with the
best mean CV accuracy, and only then check it against a held-out test set that was
never used for the decision.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

l2_grid = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]
cv_means = []
for l2 in l2_grid:
    fit_predict = lambda X_tr, y_tr, X_val, l2=l2: logistic_fit_predict(X_tr, y_tr, X_val, l2=l2)
    s = cross_val_score(X_dev, y_dev, k=5, fit_predict_fn=fit_predict)
    cv_means.append(s.mean())
    print(f'l2={l2:<8} CV accuracy={s.mean():.4f}')

best_l2 = l2_grid[int(np.argmax(cv_means))]
print(f'\nBest l2 (by CV): {best_l2}')

final_pred = logistic_fit_predict(X_dev, y_dev, X_test, l2=best_l2)
print(f'Held-out test accuracy with best l2: {np.mean(final_pred == y_test):.4f}')

plt.figure(figsize=(6, 4))
plt.plot(l2_grid, cv_means, marker='o')
plt.xscale('symlog', linthresh=0.001)
plt.xlabel('L2 strength'); plt.ylabel('Mean CV Accuracy')
plt.title('Hyperparameter Selection via 5-Fold CV')
plt.tight_layout(); plt.show()

## Sanity check against sklearn

In [ ]:
sk_pipe = make_pipeline(StandardScaler(), SklearnLR(max_iter=1000))
sk_scores = sklearn_cv_score(sk_pipe, X, y, cv=5)

print(f'sklearn 5-fold accuracy      : {sk_scores.mean():.4f}')
print(f'from-scratch 5-fold accuracy : {cross_val_score(X, y, k=5, fit_predict_fn=logistic_fit_predict).mean():.4f}')